# Feature selection — forecast t+1 (`prod_pet`)

Notebook hermano de `03_modeling_pet.ipynb`. El orden es el que pediste y el que
tiene sentido metodológico:

1. **Primero se tunean los hiperparámetros** (CV temporal, expanding window) →
   obtenemos un modelo campeón bien ajustado.
2. **Después se hace feature selection** *sobre ese campeón*: se rankean las
   features por *permutation importance* en **val** y se busca el set más chico
   que no pierde RMSE.

**Anti-leakage (ADR-028):** se tunea/entrena en `train`, se rankea y selecciona en
`val` (el mismo set que ya se usa para comparar modelos); **`test` queda intacto**.
Medir la importancia en val (no en train) evita premiar features que el modelo
memorizó pero no generalizan.

La lógica reutilizable vive en `ml/modeling.py` (tuning) y `ml/selection.py`
(selección); acá solo se orquesta.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from ml import modeling, selection

pd.set_option("display.width", 120)

## 1. Dataset y matriz de features

`build_feature_matrix` arma el dataset (features del mes `t` + target `y_next` del
mes `t+1`) y devuelve las features **crudas** (el one-hot lo hace el `Pipeline`,
por fold). Features = todas las columnas menos las claves y `y_next`.

In [ ]:
ds_enc, feature_cols = modeling.build_feature_matrix()  # target='prod_pet' por defecto

print(f"Filas: {len(ds_enc):,}  |  Features: {len(feature_cols)}")
print(ds_enc["split"].value_counts().reindex(["train", "val", "test"]).to_string())

## 2. Split train / val

`X` queda como DataFrame con las columnas crudas; el `Pipeline` se encarga del
one-hot / imputación / escalado ajustando **solo en train**. Guardamos también
`X_train`/`X_val` porque los reusa la feature selection.

In [ ]:
X_train, y_train, X_val, y_val = modeling.split_train_val(ds_enc, feature_cols)
print("train:", X_train.shape, " | val:", X_val.shape)

## 3. Búsqueda de hiperparámetros (primero) — CV temporal

`tune_all` corre un **random search** con **CV temporal (expanding window por mes)**:
en cada fold el bloque de validación es posterior a todo su train → sin leakage
futuro→pasado. El preprocesamiento va dentro del `Pipeline`, así que se reajusta
por fold. La búsqueda es **dentro de train**; `val` sigue intacto para la
comparación final. Modelos: **Ridge**, **Random Forest**, **XGBoost**.

> ⚠️ Entrena muchos modelos: puede tardar varios minutos. Cada modelo muestra
> una barra de progreso con el mejor RMSE de CV hasta el momento.

In [ ]:
tabla_cv, searches = modeling.tune_all(ds_enc, feature_cols, n_splits=4)
tabla_cv

### 3.1 Evaluación de los tuneados en val + elección del campeón

Cada `best_estimator_` (reentrenado en todo train con sus mejores hiperparámetros)
se evalúa sobre **val**. Elegimos como **campeón** el de menor `val_rmse` (excluyendo
la persistencia, que es solo la referencia a batir).

In [ ]:
comparacion_tuned = modeling.eval_searches_on_val(searches, ds_enc, feature_cols)
comparacion_tuned.round(3)

In [ ]:
nombre_campeon = min(
    searches,
    key=lambda n: modeling.evaluate(y_val, searches[n].best_estimator_.predict(X_val))["val_rmse"],
)
champion = searches[nombre_campeon].best_estimator_
print(f"Campeón (menor val_rmse): {nombre_campeon}")
print("Hiperparámetros:", searches[nombre_campeon].best_params_)

## 4. Feature selection (sobre el campeón)

### 4.1 Ranking por *permutation importance* en val

Para cada feature se **permuta su columna en val** (`n_repeats` veces) y se mide
cuánto sube el RMSE. Como el `Pipeline` hace el one-hot adentro, se permuta a nivel
de **feature cruda** (p. ej. `empresa` entera, no una categoría suelta).

La importancia queda en **m³** = cuánto empeora el RMSE de val al romper la feature.
**> 0** = útil; **≈ 0 o < 0** = ruido (permutarla no empeora, o hasta mejora).

In [ ]:
ranking = selection.permutation_ranking(champion, X_val, y_val, n_repeats=10)
ranking.round(2)

Visualización del ranking (las 20 features más importantes; barra = aumento de
RMSE al permutar, con su desvío entre repeticiones):

In [ ]:
import matplotlib.pyplot as plt

top = ranking.head(20).iloc[::-1]  # la más importante, arriba
fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(top["feature"], top["imp_mean"], xerr=top["imp_std"], color="#4C72B0")
ax.axvline(0, color="0.6", lw=0.8)
ax.set_xlabel("Importancia = ↑ RMSE de val al permutar (m³)")
ax.set_title(f"Permutation importance en val — {nombre_campeon}")
fig.tight_layout()
plt.show()

### 4.2 Curva top-k: RMSE en val según nº de features

Reentrenamos el campeón (mismos hiperparámetros) usando solo las **top-k** features
del ranking. Buscamos la **rodilla**: a partir de cierto `k`, sumar features ya no
baja el RMSE → nos podemos quedar con el set chico.

In [ ]:
curva = selection.evaluate_topk(ranking, champion, X_train, y_train, X_val, y_val)
curva.round(3)

### 4.3 Set seleccionado y comparación completo vs reducido

`select_features` se queda con las features de importancia **> 0**. Comparamos el
campeón **completo** vs **reducido** (idénticos hiperparámetros) en val: si el RMSE
se mantiene con muchas menos features, la reducción vale la pena (modelo más simple,
más rápido y con menos features que mantener en el feature store).

In [ ]:
seleccionadas = selection.select_features(ranking)  # imp_mean > 0
print(f"Seleccionadas ({len(seleccionadas)}/{len(feature_cols)}):")
print(seleccionadas)

In [ ]:
_, m_full = selection.refit_and_eval(champion, feature_cols, X_train, y_train, X_val, y_val)
_, m_red = selection.refit_and_eval(champion, seleccionadas, X_train, y_train, X_val, y_val)

pd.DataFrame([
    {"modelo": f"{nombre_campeon} (completo)", "n_features": len(feature_cols), **m_full},
    {"modelo": f"{nombre_campeon} (reducido)", "n_features": len(seleccionadas), **m_red},
]).round(3)

## 5. Conclusión / próximos pasos

- Si el modelo **reducido** empata (o casi) al completo, nos quedamos con el set
  chico y lo documentamos en un **ADR** (baja de features + criterio de selección
  leakage-safe).
- Opcional pero recomendado: **re-tunear** los hiperparámetros con el set reducido
  (el óptimo puede moverse un poco al cambiar la dimensionalidad).
- Recordá el objetivo de fondo: dejar features **autorregresivas del propio target**
  para habilitar el forecast recursivo (bajar `prod_vecinos_mean` y las que dependen
  de otras series medidas del pozo, como `water_cut`). Esta selección es el primer
  filtro empírico de esa decisión.